# 02 — Preprocessing

Load the pseudobulk data from Notebook 01 and apply QC filters:

1. Remove low-expression genes
2. Remove housekeeping / ribosomal / mitochondrial genes
3. Log-transform expression values

**Inputs:** `data/processed/pseudobulk_mean_cpm.parquet`, `pseudobulk_detection_rate.parquet`  
**Outputs:** `data/processed/preprocessed_mean_log2cpm.parquet`, `preprocessed_detection_rate.parquet`

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

from src.data_loader import load_pseudobulk
from src.preprocessing import (
    filter_low_expression,
    remove_housekeeping,
    log1p_transform,
    run_preprocessing,
    flag_housekeeping_and_ribosomal,
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"

## 1. Load Pseudobulk Data

In [ ]:
mean_df, det_df = load_pseudobulk(DATA_DIR)
print(f"Raw pseudobulk: {mean_df.shape[0]:,} genes × {mean_df.shape[1]} cell types")
print(f"\nExpression range: {mean_df.values.min():.2f} — {mean_df.values.max():.2f} CPM")
print(f"Median non-zero expression: {mean_df.values[mean_df.values > 0].flatten().min():.4f} CPM")

## 2. Overview Before Filtering

In [ ]:
# How many genes are expressed in at least one cell type?
n_expressed = (mean_df > 0).any(axis=1).sum()
n_zero = (mean_df == 0).all(axis=1).sum()
print(f"Genes with >0 expression in ≥1 cell type: {n_expressed:,}")
print(f"Genes with zero expression everywhere:     {n_zero:,}")

# Housekeeping / ribosomal / MT gene counts
hk_mask = flag_housekeeping_and_ribosomal(mean_df.index)
print(f"\nHousekeeping/ribosomal/MT genes:           {hk_mask.sum():,}")

## 3. Run Preprocessing Pipeline

Steps:
- Remove genes with mean CPM < 1 and detection rate < 1% in all cell types
- Remove housekeeping, ribosomal protein, and mitochondrial genes
- Log2(1 + CPM) transform

In [ ]:
mean_proc, det_proc = run_preprocessing(
    mean_df, det_df,
    min_cpm=1.0,
    min_det_rate=0.01,
    remove_hk=True,
    log_transform=True,
)
print(f"\nAfter preprocessing: {mean_proc.shape[0]:,} genes × {mean_proc.shape[1]} cell types")

## 4. Summary Statistics After Filtering

In [ ]:
print("Gene count at each filtering step:")
print(f"  Raw:                  {mean_df.shape[0]:>6,}")

m1, d1 = filter_low_expression(mean_df, det_df, min_cpm=1.0, min_det_rate=0.01)
print(f"  After low-expr:       {m1.shape[0]:>6,}")

m2, d2 = remove_housekeeping(m1, d1)
print(f"  After HK removal:     {m2.shape[0]:>6,}")
print(f"  After log transform:  {mean_proc.shape[0]:>6,} (same count, values changed)")

In [ ]:
# Expression distribution after preprocessing
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of max expression per gene
max_expr = mean_proc.max(axis=1)
axes[0].hist(max_expr, bins=100, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Max log2(1+CPM) across cell types")
axes[0].set_ylabel("Number of genes")
axes[0].set_title("Distribution of max expression per gene")

# Distribution of number of cell types expressing each gene
n_types_expressing = (det_proc > 0.01).sum(axis=1)
axes[1].hist(n_types_expressing, bins=50, color="coral", edgecolor="white")
axes[1].set_xlabel("Number of cell types with det rate > 1%")
axes[1].set_ylabel("Number of genes")
axes[1].set_title("Gene expression breadth")

plt.tight_layout()
plt.show()

## 5. Save Preprocessed Data

Save both the log-transformed expression and the raw CPM (needed for fold-change).

In [ ]:
# Log-transformed (for Tau and visualisation)
mean_proc.to_parquet(DATA_DIR / "preprocessed_mean_log2cpm.parquet")
det_proc.to_parquet(DATA_DIR / "preprocessed_detection_rate.parquet")

# Also save the filtered but UN-logged CPM (for fold-change computation)
# Re-derive from raw by applying same gene filter
kept_genes = mean_proc.index
mean_cpm_filtered = mean_df.loc[kept_genes]
mean_cpm_filtered.to_parquet(DATA_DIR / "preprocessed_mean_cpm.parquet")

print(f"Saved preprocessed data to {DATA_DIR}")
print(f"  - preprocessed_mean_log2cpm.parquet  ({mean_proc.shape[0]:,} genes)")
print(f"  - preprocessed_mean_cpm.parquet       (raw CPM, same genes)")
print(f"  - preprocessed_detection_rate.parquet")

## Summary

Preprocessing complete. Next step: **Notebook 03 — Specificity Analysis**.